In [1]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_vertexai import ChatVertexAI

In [3]:
policy_prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "you are an HR policy writer creating policies"
     "you will recieve an HR POLICY Template which is extracted from a DOCX file.\n"
     "Generate a New policy that follows same structure/headings style, but write ORIGINAL content.\n"
     "Rules:\n"
     "- Do not copy long phrases verbatiam from the template"
     "- Generate a concise and Well-structure policy"),
     ("user", 
      "TEMPLATE (reference): \n---\n{template_text}"
      "Generate policy:\n"
      "- company: {company_name}\n"
      "- company size: {company_size}\n"
      "- tone: {tone} \n"
      "- company category: {company_category}\n"
      "- country context: {country}\n"
      "Return only Markdown")

])

In [46]:
loader = Docx2txtLoader("../docs/Company Policies/Attendance Policy.docx")
docs = loader.load()
template_text = "\n\n".join(doc.page_content for doc in docs)


In [47]:
print(template_text)

Attendance Policy



1. OVERVIEW



Each employee at [Company Name] is responsible for punctual and consistent attendance. Employees should arrive on time, be prepared to work, and on schedule. Employees are also expected to stay at work for the whole of their shift. It is inconvenient to arrive late, leave early, or miss other scheduled hours, and it must be avoided.



This policy does not apply to FMLA-covered (FMLA - Family and Medical Leave Act) absences or leave taken as a reason

able accommodation under the Americans with Disabilities Act (ADA). They have their own policies that cover these exceptions.



2. OBJECTIVE

The goal of this policy is to lay out [Company Names] policies and processes for dealing with employee absences and tardiness in order to increase the company's efficiency and reduce unscheduled absences.



3. ATTENDANCE INFRACTIONS CALCULATION



Absent with calls - 1 Point 

Absent with no calls - 2 Points 

Tardy - ½ Point

Early Departure - ½ Point

Returnin

In [17]:
llm = ChatVertexAI(
    model_name="gemini-2.5-flash-lite",
    temperature=0.2,
    max_output_tokens=500
)

C:\Users\shala\AppData\Local\Temp\ipykernel_22616\625557697.py:1: DeprecationWarning: Use [`ChatGoogleGenerativeAI`][langchain_google_genai.ChatGoogleGenerativeAI] instead.
  llm = ChatVertexAI(


In [ ]:
chain = policy_prompt | llm
response = chain.invoke(
    {
        "template_text": template_text,
        "company_name": "Acme Corp",
        "company_size": "10000 employees",
        "tone": "professional",
        "company_category": "Information Technology",
        "country": "India"
    })

In [21]:
response.pretty_print()

================================== Ai Message ==================================

---
**Acme Corp - Employee Punctuality and Attendance Policy**

**1. Introduction**

Acme Corp values the dedication and commitment of its employees. Punctual and consistent attendance is crucial for maintaining operational efficiency and ensuring seamless collaboration within our Information Technology environment. All employees are expected to report for duty on time, be ready to commence work at their scheduled start time, and remain engaged throughout their entire shift. Disruptions caused by late arrivals, early departures, or unscheduled absences can significantly impact team productivity and project timelines.

This policy is not intended to cover absences that fall under legally protected leave entitlements, such as those mandated by the Maternity Benefit Act, the Employees' State Insurance Act, or other applicable Indian labor laws, nor does it supersede policies related to reasonable accommodati

In [51]:
from langchain_community.document_loaders import DirectoryLoader
directory_loader = DirectoryLoader(
    "../docs/Company Policies",
    glob="*.docx",
    loader_cls=Docx2txtLoader
)
files = directory_loader.load()
files[0]

Document(metadata={'source': '..\\docs\\Company Policies\\AI Tool Usage Policy.docx'}, page_content="AI Tool Usage Policy\n\nPurpose\n\nThis AI Tool Usage Policy outlines the guidelines and procedures for the responsible and ethical use of artificial intelligence (AI) tools within the organization. The policy is designed to ensure that AI technologies are employed in a manner that aligns with the organization's values, legal requirements, and data protection standards.\n\nScope\n\nThis policy applies to all employees, contractors, and third parties who have access to and use AI tools on behalf of the organization.\n\nEthical Use of AI\n\nFairness: AI tools should be programmed and utilized in a manner that promotes fairness and avoids bias in decision-making processes.\n\nTransparency: The organization commits to being transparent about the use of AI tools, providing clear explanations of how AI systems make decisions that impact individuals.\n\nAccountability: There should be clear ac

In [39]:
import os

def combine_dir_with_markdown(dir_path, docx_path):
    # Extract filename from docx path
    filename = os.path.basename(docx_path)
    
    # Remove extension and convert to markdown name
    name_without_ext = os.path.splitext(filename)[0]
    markdown_name = name_without_ext.replace(" ", "_") + ".md"
    
    # Combine directory path with markdown filename
    return os.path.join(dir_path, markdown_name)

In [ ]:
from langchain_community.document_loaders import DirectoryLoader
#chain = policy_prompt | llm  uncomment to run the llm

directory_loader = DirectoryLoader(
    "../docs/Policy/CompanyPolicies",
    glob="*.docx",
    loader_cls=Docx2txtLoader)

docs = directory_loader.load()
for doc in docs:
    #template_text = "\n\n".join(doc.page_content for doc in docs) # this need to be corrected
    template_text = doc.page_content
    path = combine_dir_with_markdown(
        "../generated_data", doc.metadata['source'])
    response = chain.invoke(
        {
            "template_text": template_text,
            "company_name": "Acme Corp",
            "company_size": "10000 employees",
            "tone": "professional",
            "company_category": "Information Technology",
            "country": "India"
        })
    with open(path, 'w') as f:
        f.write(response.content)

117